[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/humanization/humanization_guide/09_developability/09_developability_lab.ipynb)


# 09 — Developability — liability 모티프 직접 스캔

> 본문 [`09_developability.md`](09_developability.md) 와 **한 절씩 짝지어** 보세요.
> **실측 소요 —** 정규식 liability 스캔(후보 5종) **1초 미만**

## 0) Colab 준비 — 저장소 클론 & 작업 폴더 이동

이 셀이 저장소를 클론하고 `09_developability/` 로 이동해 필요한 패키지만 깝니다(로컬에서 `09_developability/` 안에 열었다면 클론 생략).
끝나면 **`my_run/`**(내가 만들 결과)과 **`data/`**(커밋된 레퍼런스)가 준비돼요 — 랩은 `my_run/` 을 먼저 찾고 없으면 `data/` 로 폴백하며 어느 쪽을 썼는지 매번 print 합니다.

In [ ]:
# ====== Colab/로컬 공용 부트스트랩 (모든 챕터 공통) ======
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # 이 가이드 저장소 (fork 했다면 본인 주소로 바꾸세요)
CLONE_AS = "bio-guides"
CHAPTER  = "09_developability"
APT_PKGS = ""     # Colab 에서만: 시스템 패키지 (hmmer = ANARCI 가 부르는 hmmscan)
PIP_PKGS = "pandas matplotlib"     # 없는 것만 설치

import os, sys, json, time, shutil, pathlib, subprocess, importlib, importlib.util
IN_COLAB = "google.colab" in sys.modules

# HuggingFace 가중치 다운로드가 '멈춘 채' 매달리는 일을 막아요.
# (멈춤은 예외가 안 나서 폴백이 안 걸려요 — 타임아웃을 걸어 실패로 바꿔야 data/ 로 이어져요)
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")   # 스트림 30초 무응답 → 끊고 재시도
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "15")

def _run(cmd, check=True, timeout=None):
    # timeout 을 꼭 주세요 — 네트워크가 '멈춘 채' 매달리면 예외가 안 나서 data/ 폴백이 안 걸립니다.
    print("$", cmd)
    return subprocess.run(cmd, shell=True, check=check, timeout=timeout)

_MARK = "humanization_viz.py"          # 이 파일이 있는 폴더 = 가이드 루트

def _find_root():
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    # 클론 직후: cwd '아래'만 깊이 3까지 훑는다.
    # (상위로 올라가 rglob 하면 Colab 에서는 / 전체를 뒤지게 된다 — 매우 느리고 엉뚱한 경로를 잡는다)
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "가이드 루트를 못 찾았어요. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

GUIDE_ROOT = ROOT.resolve()
os.chdir(GUIDE_ROOT / CHAPTER)         # data/ 상대경로가 그대로 동작
sys.path.insert(0, str(GUIDE_ROOT))    # humanization_viz import
sys.path.insert(0, str(GUIDE_ROOT / CHAPTER))

_IMPORT_NAME = {"biopython": "Bio", "pyyaml": "yaml", "scikit-learn": "sklearn"}

def _have(mod):
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False

def _ensure(pkgs):
    miss = [p for p in pkgs.split() if not _have(_IMPORT_NAME.get(p, p.replace("-", "_")))]
    if miss:
        print("설치:", " ".join(miss))
        _run(f'"{sys.executable}" -m pip -q install ' + " ".join(miss))
        importlib.invalidate_caches()

_APT = APT_PKGS.split() if (APT_PKGS and IN_COLAB) else []   # 모아서 apt 를 한 번만 돌립니다
if PIP_PKGS:
    _ensure(PIP_PKGS)

def _ensure_pkg_resources():
    # setuptools 81+(2026-02) 이 pkg_resources 를 없앴는데 IgFold 의존성이 이걸 import 합니다.
    if importlib.util.find_spec("pkg_resources") is None:
        _run(f'"{sys.executable}" -m pip -q install "setuptools<81"')
        importlib.invalidate_caches()

import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    _APT.append("fonts-nanum")             # Colab 엔 한글 폰트가 없어 라벨이 □ 로 깨져요

if _APT:                                   # apt 인덱스 갱신은 한 번만 (ANARCI 는 hmmscan 이 필요해요)
    _run("apt-get -qq update", timeout=600)
    _run("apt-get -qq install -y " + " ".join(_APT), timeout=900)


# ── ① 내가 만든 결과(my_run) 우선 · ② 없으면 커밋된 레퍼런스(data/) ─────────────
MY  = pathlib.Path("my_run"); MY.mkdir(exist_ok=True)
REF = pathlib.Path("data")

def find_one(pattern, ref=REF, quiet=False):
    """산출물 하나 찾기 — 내가 만든 my_run/ 을 먼저 뒤지고, 없으면 커밋된 레퍼런스에서."""
    hits = sorted(MY.rglob(pattern))
    if hits:
        if not quiet: print(f"[내 결과]   {hits[0]}")
        return hits[0]
    hits = sorted(pathlib.Path(ref).glob(pattern))
    assert hits, f"'{pattern}' 을 my_run/ 에서도 {ref}/ 에서도 찾지 못했어요."
    if not quiet: print(f"[레퍼런스] {hits[0]}")
    return hits[0]

def find_prev(chapter, pattern, ref=REF, quiet=False):
    """앞 챕터에서 직접 만든 산출물 → 없으면 이 챕터 data/ 레퍼런스."""
    hits = sorted((GUIDE_ROOT / chapter / "my_run").rglob(pattern))
    if hits:
        if not quiet: print(f"[내 결과 · {chapter}] {hits[0]}")
        return hits[0]
    return find_one(pattern, ref, quiet)

def read_fasta(path):
    out, name, buf = {}, None, []
    for line in pathlib.Path(path).read_text().splitlines():
        if line.startswith(">"):
            if name: out[name] = "".join(buf)
            name, buf = line[1:].strip(), []
        elif line.strip():
            buf.append(line.strip())
    if name: out[name] = "".join(buf)
    return out

def write_fasta(path, records):
    pathlib.Path(path).write_text("".join(f">{k}\n{v}\n" for k, v in records.items()))
    return pathlib.Path(path)

PARENTAL = read_fasta(REF / "parental.fasta")      # 가이드 전체를 관통하는 러닝 예제
VH = PARENTAL["parental_H"]
VL = PARENTAL["parental_L"]

print("가이드 루트 :", GUIDE_ROOT)
print("작업 폴더   :", pathlib.Path.cwd())
print(f"러닝 예제   : VH {len(VH)} aa · VL {len(VL)} aa (mouse hybridoma 가정)")

## 1) 직접 실행 — 서열 한 줄로 잡히는 liability (본문 9.1)

liability 는 한 종류가 아니에요. **서열만으로 스캔되는 것**과 **구조가 있어야 잡히는 것**으로 갈려요. 이 절은 앞쪽만 다뤄요.

| 서열로 잡히는 것 | 방법 | 위험 |
|---|---|---|
| N-glycosylation | `N[^P][ST]` | 예상치 못한 당쇄 → 이질성·클리어런스 |
| deamidation | `N[GS]` | 보관 중 전하 변이 |
| isomerization | `DG` | 구조 변형 |
| oxidation | `[MW]` | 산화 → 활성 저하 |
| unpaired Cys | Cys 개수의 홀짝 | 미스폴딩·이량체 |

`NXS/T` 의 X 가 P 면 당쇄가 안 붙어요 — 정규식이 `N[^P][ST]` 인 이유예요.
SAP·charge patch 는 Ch.08 의 예측 구조가 있어야 하니 5절에서 따로 봐요.

In [ ]:
import re, json, difflib, pandas as pd

MOTIFS = {
    "N-glycosylation": r"N[^P][ST]",   # NXS/T (X != P)
    "deamidation":     r"N[GS]",
    "isomerization":   r"DG",
    "oxidation":       r"[MW]",
}

def scan(seq):
    """서열 → {모티프: [1-based 위치, ...]}"""
    return {name: [m.start() + 1 for m in re.finditer(p, seq)] for name, p in MOTIFS.items()}

PREV = [
    ("05_humanize_sapiens", "sapiens_humanized_noguard.fasta", ("sapiens_humanized_VH", "sapiens_humanized_VL"), "sapiens"),
    ("06_cdr_safe_tools",   "humatch_humanised.fasta",         ("humatch_humanised_VH", "humatch_humanised_VL"), "humatch"),
    ("06_cdr_safe_tools",   "anthroab_best_score.fasta",       ("anthroab_bestscore_VH", "anthroab_bestscore_VL"), "anthroab"),
    ("06_cdr_safe_tools",   "anthroab_masked_FRonly.fasta",    ("anthroab_masked_VH", "anthroab_masked_VL"), "anthroabFRmasked"),
]

cands, SRC = {"parental": (VH, VL)}, {"parental": "data/parental.fasta"}
for chapter, fname, keys, label in PREV:
    p = GUIDE_ROOT / chapter / "my_run" / fname
    if p.exists():
        f = read_fasta(p)
        cands[label], SRC[label] = (f[keys[0]], f[keys[1]]), f"내 결과 · {chapter}/my_run/{fname}"

need = [lab for *_, lab in PREV if lab not in cands]          # 빠진 후보만 레퍼런스로 채워요
if need:
    vp = find_one("variants.fasta", quiet=True)
    v = read_fasta(vp)
    for lab in need:
        cands[lab], SRC[lab] = (v[f"{lab}_VH"], v[f"{lab}_VL"]), f"레퍼런스 · {vp}"

for k in cands:
    print(f"  {k:18s} ← {SRC[k]}")

# unpaired Cys — 개수가 홀수면 짝 없는 시스테인이 남아요
print()
odd = []
for name, (h, l) in cands.items():
    nh, nl = h.count("C"), l.count("C")
    if nh % 2 or nl % 2:
        odd.append(name)
    print(f"  {name:18s} Cys VH {nh}개 · VL {nl}개 · {'짝 맞음' if not (nh % 2 or nl % 2) else '★홀수 — unpaired Cys'}")
print(f"\nunpaired Cys 판정 — 홀수 후보 {len(odd)}개"
      + ("" if odd else ". 전부 짝이 맞아 미스폴딩·이량체 위험은 이 축에서 걸리지 않아요."))

## 2) 내 결과 확인 — 후보별 모티프 총량 (본문 9.2)

8개(체인 2 × 모티프 4) 조합을 다 늘어놓으면 **후보 전부 0인 칸**이 섞여요. 0만 있는 칸은 정보가 아니라 잡음이라서 표에서 빼고, 어떤 칸을 뺐는지만 알려요.

In [ ]:
rows = []
for name, (h, l) in cands.items():
    for chain, seq in (("VH", h), ("VL", l)):
        for motif, hits in scan(seq).items():
            rows.append({"candidate": name, "chain": chain, "motif": motif,
                         "count": len(hits), "positions_1based": ";".join(map(str, hits))})
lia = pd.DataFrame(rows)
lia.to_csv(MY / "liability.csv", index=False)
print("→", MY / "liability.csv")

piv = lia.pivot_table(index="candidate", columns=["chain", "motif"], values="count",
                      aggfunc="sum", fill_value=0)
piv = piv.reindex(list(cands))
zero = [col for col in piv.columns if piv[col].sum() == 0]
print("후보 전부 0건이라 뺀 칸:", ", ".join(f"{a} {b}" for a, b in zero) if zero else "없음")
display(piv.drop(columns=zero))

tot = lia.groupby("candidate")["count"].sum().reindex(list(cands))
ox  = lia[lia.motif == "oxidation"].groupby("candidate")["count"].sum().reindex(list(cands)).fillna(0)
print()
for name in cands:
    print(f"  {name:18s} 총 {int(tot[name]):2d}건 · 그중 oxidation {int(ox[name]):2d}건 ({ox[name]/tot[name]*100:.0f}%)")
print(f"\n판정 — oxidation 비중이 {ox.div(tot).min()*100:.0f}~{ox.div(tot).max()*100:.0f}% 로 전 후보가 비슷해요.")
print("Met/Trp 은 어느 항체 서열에나 흔해서 총량으로는 후보가 갈리지 않아요.")
print("갈라지는 축은 총량이 아니라 parental 대비 신규 모티프예요 (3절).")

In [ ]:
from humanization_viz import liability_overview
from IPython.display import Image, display

png = liability_overview(lia.groupby(["candidate", "motif"], sort=False)["count"].sum().reset_index(),
                         "Developability liability motifs (VH+VL)", str(MY / "09_liability.png"))
display(Image(str(png)))
worst = tot.idxmax()
print(f"막대 높이 판정 — 가장 높은 후보는 {worst} ({int(tot[worst])}건), 가장 낮은 후보는 "
      f"{tot.idxmin()} ({int(tot.min())}건). parental 은 {int(tot['parental'])}건이에요.")
print("총량이 parental 보다 크게 늘어난 후보가 developability 축의 1차 경고 대상이에요.")

## 3) 증분 스캔 — parental 에 없던 자리만 (본문 9.2)

원래 있던 모티프는 parental 항체가 이미 견디던 자리예요. 진짜 위험은 humanization 이 **새로 심은 자리**예요.
그중에서도 신규 `NXS/T` 가 제일 위험해요 — 예상치 못한 당쇄는 이질성·클리어런스로 바로 이어지거든요.

In [ ]:
par_scan = {"VH": scan(cands["parental"][0]), "VL": scan(cands["parental"][1])}

new_rows = []
for name, (h, l) in cands.items():
    if name == "parental":
        continue
    for chain, seq in (("VH", h), ("VL", l)):
        for motif, hits in scan(seq).items():
            base = set(par_scan[chain][motif])
            new, lost = sorted(set(hits) - base), sorted(base - set(hits))
            new_rows.append({"candidate": name, "chain": chain, "motif": motif,
                             "신규": len(new), "신규 위치": ";".join(map(str, new)) or "-",
                             "사라짐": len(lost)})
delta = pd.DataFrame(new_rows)
delta.to_csv(MY / "liability_delta.csv", index=False)

display(delta[delta["신규"] > 0].sort_values(["candidate", "chain"]).reset_index(drop=True))

new_tot = delta.groupby("candidate")["신규"].sum().reindex([n for n in cands if n != "parental"])
glyc = delta[(delta.motif == "N-glycosylation") & (delta["신규"] > 0)]
print()
for name, v_ in new_tot.items():
    print(f"  {name:18s} 신규 {int(v_):2d}건")
print(f"\n신규 N-glycosylation 모티프 — 총 {int(glyc['신규'].sum())}건", end="")
if glyc.empty:
    print(". 이 축에서는 아무도 걸리지 않았어요.")
else:
    print(" ·", ", ".join(f"{r.candidate} {r.chain} {r['신규 위치']}" for _, r in glyc.iterrows()))
    print("판정 — 신규 NXS/T 가 있는 후보는 Ch.10 의 hard filter 대상이에요(즉시 탈락 또는 backmutation 검토).")
print(f"신규가 가장 많은 후보는 {new_tot.idxmax()} ({int(new_tot.max())}건), 가장 적은 후보는 "
      f"{new_tot.idxmin()} ({int(new_tot.min())}건) 이에요.")

## 4) CDR 안에 떨어졌나 — 좌표를 IMGT 로 다시 매핑 (본문 9.2)

같은 모티프라도 **CDR 안**이면 결합에 직접 영향을 줘서 위험도가 달라요.
그런데 CDR 좌표는 parental raw 번호 기준이고, **indel 이 있는 후보는 raw 번호가 밀려요.**
그래서 Ch.04 가 만든 `raw2imgt_*.json` 으로 parental 좌표를 **IMGT 라벨**로 바꾸고, 후보 좌표도 같은 라벨로 옮긴 뒤에 비교해요.

In [ ]:
def load_r2i(tag):
    p = find_prev("04_sequence_qc", f"raw2imgt_{tag}.json",
                  ref=GUIDE_ROOT / "04_sequence_qc" / "data", quiet=True)
    return {int(k): v for k, v in json.loads(pathlib.Path(p).read_text()).items()}

r2i = {"H": load_r2i("H"), "L": load_r2i("L")}
print("raw → IMGT 크로스워크 — VH", len(r2i["H"]), "자리 · VL", len(r2i["L"]), "자리")

# parental raw CDR 구간 → IMGT 라벨 집합 (여기가 '보호 좌표'의 기준면)
ct = pd.read_csv(find_one("cdr_table_imgt.csv", quiet=True))
guard = {"H": {}, "L": {}}
for _, r in ct.iterrows():
    seq = VH if r["chain"] == "H" else VL
    st = seq.find(r["sequence"]) + 1
    assert st > 0, f"CDR {r['cdr']} 를 parental 서열에서 못 찾았어요 — cdr_table 과 parental.fasta 가 어긋나요."
    for p in range(st, st + len(r["sequence"])):
        guard[r["chain"]][r2i[r["chain"]][p]] = f"{r['chain']}-{r['cdr']}"

def cand2imgt(par, cand, tag):
    """후보 raw 위치 → IMGT 라벨. 길이가 같으면 1:1, 다르면(indel) 정렬로 이어 붙여요."""
    if len(par) == len(cand):
        return {i + 1: r2i[tag][i + 1] for i in range(len(cand))}
    m = {}
    for op, i1, i2, j1, j2 in difflib.SequenceMatcher(None, par, cand, autojunk=False).get_opcodes():
        if op in ("equal", "replace"):
            for k in range(min(i2 - i1, j2 - j1)):
                m[j1 + k + 1] = r2i[tag][i1 + k + 1]
    return m

hits, shifted = [], []
for name, (h, l) in cands.items():
    for chain, seq, tag in (("VH", h, "H"), ("VL", l, "L")):
        par = VH if tag == "H" else VL
        m = cand2imgt(par, seq, tag)
        if len(par) != len(seq):
            moved = [p for p, lab in m.items() if r2i[tag].get(p) != lab]
            shifted.append((name, chain, len(par), len(seq), len(m), len(moved)))
        for motif, ps in scan(seq).items():
            for p in ps:
                lab = m.get(p)
                if lab in guard[tag]:
                    hits.append({"candidate": name, "chain": chain, "motif": motif,
                                 "raw": p, "IMGT": lab, "구간": guard[tag][lab]})

cdr_lia = pd.DataFrame(hits)
display(cdr_lia if not cdr_lia.empty else "CDR 안 liability 없음")

for name, chain, lp, lc, nmap, nmoved in shifted:
    print(f"재매핑 — {name} {chain} 은 길이 {lp}→{lc} 라 정렬로 이어 붙였어요 "
          f"(잇힌 자리 {nmap}개 · 그중 raw 번호가 실제로 밀린 자리 {nmoved}개).")
if not shifted:
    print("재매핑 — indel 있는 후보가 없어 raw 번호가 그대로 IMGT 로 이어졌어요.")

par_cdr = set(map(tuple, cdr_lia[cdr_lia.candidate == "parental"][["chain", "motif", "IMGT"]].values)) if not cdr_lia.empty else set()
new_in_cdr = [r for _, r in cdr_lia.iterrows()
              if r.candidate != "parental" and (r.chain, r.motif, r.IMGT) not in par_cdr] if not cdr_lia.empty else []
print(f"\n판정 — CDR 안 모티프 {len(cdr_lia)}건 중 parental 에 없던 신규는 {len(new_in_cdr)}건이에요.")
for r in new_in_cdr:
    print(f"  ★ {r.candidate} {r['구간']} {r.IMGT} 에 신규 {r.motif}")
print("parental 부터 갖고 있던 자리는 이미 견디던 자리라 우선순위가 낮고, 신규만 backmutation 후보로 올려요.")

## 5) 구조가 있어야 잡히는 것 — SAP · charge patch · TAP (본문 9.1 심화 · 9.3)

여기까지가 서열 스캔이에요. 응집·점도·클리어런스로 이어지는 지표는 **Ch.08 의 예측 구조 위에서** 돌아가요.

In [ ]:
struct_based = pd.DataFrame([
    {"지표": "SAP (Spatial Aggregation Propensity)", "필요한 입력": "예측 구조 + 표면 노출·소수성", "이 랩": "미실행"},
    {"지표": "charge patch / pI 비대칭",             "필요한 입력": "예측 구조 + 표면 전하 분포",   "이 랩": "미실행"},
    {"지표": "TAP (Therapeutic Antibody Profiler)",  "필요한 입력": "예측 구조 + 임상단계 항체 분포", "이 랩": "미실행 · 웹 전용"},
])
display(struct_based)
print("세 지표는 예측 구조의 품질을 그대로 물려받아요.")
print("그래서 절대 기준선으로 자르지 말고, 같은 파이프라인으로 뽑은 후보끼리 상대 비교로 읽어요.")
print("반대로 1~4절의 정규식 스캔은 구조 품질과 무관해서 절대값으로 읽어도 됩니다 — 이 차이가 두 축을 나눠 쓰는 이유예요.")

## 6) 레퍼런스 대조 (본문 9.2)

`data/liability_reference.csv` 는 이 가이드를 만들 때 같은 정규식으로 뽑은 산출물이에요. 개수뿐 아니라 **위치 문자열까지** 맞춰 봐요.

In [ ]:
ref = pd.read_csv(find_one("liability_reference.csv", quiet=True)).fillna({"positions_1based": ""})
cmp_ = lia.merge(ref, on=["candidate", "chain", "motif"], suffixes=("_my", "_ref"))
ok_cnt = bool((cmp_.count_my == cmp_.count_ref).all())
ok_pos = bool((cmp_.positions_1based_my == cmp_.positions_1based_ref).all())
print(f"대조한 행 {len(cmp_)}개 · 개수 일치 {ok_cnt} · 위치 일치 {ok_pos}")

ref_tot = ref.groupby("candidate")["count"].sum().reindex(list(cands))
print("\n[레퍼런스 총량]")
for name in cands:
    print(f"  {name:18s} {int(ref_tot[name]):2d}건")
ref_pos = {(r.candidate, r.chain): set(str(r.positions_1based).split(";")) - {""}
           for r in ref[ref.motif == "N-glycosylation"].itertuples()}
ref_new_glyc = sorted({cnd for (cnd, chain) in ref_pos if cnd != "parental"
                       and ref_pos[(cnd, chain)] - ref_pos[("parental", chain)]})
print("레퍼런스에서 신규 N-glyc 를 만든 후보 —", ", ".join(ref_new_glyc) or "없음")

if ok_cnt and ok_pos:
    print("\n판정 — 커밋된 레퍼런스가 variants.fasta 에서 정규식만으로 완전히 재현돼요.")
    print("실행을 건너뛴 사람도 같은 표에 도달하니, 아래 결론은 내 결과·레퍼런스 어느 쪽으로 읽어도 같아요.")
else:
    print("\n판정 — 개수나 위치가 어긋나요. 내 후보 서열이 레퍼런스와 다른 경우(직접 만든 my_run)라면 정상이에요.")
    display(cmp_[(cmp_.count_my != cmp_.count_ref) | (cmp_.positions_1based_my != cmp_.positions_1based_ref)])

## 이 랩에서 확인한 것

1. 서열 liability 스캔은 **정규식 4줄 + Cys 홀짝 1줄**이면 끝나요 — 후보를 만들 때마다 자동으로 돌리세요.
2. 총량은 후보를 못 갈라요. **oxidation(Met/Trp)이 총량의 83~100%** 를 차지하는데, Met·Trp 은 어느 서열에나 흔하거든요.
3. 갈라지는 축은 **parental 대비 신규 모티프**예요 — 특히 신규 `NXS/T` 는 Ch.10 의 **hard filter** 로 직행해요.
4. CDR 귀속은 **IMGT 라벨로 다시 매핑한 뒤** 판단해요. raw 번호는 indel 후보에서 밀려요.
5. SAP·charge patch·TAP 는 예측 구조 위에서 돌고, **후보 간 상대 비교**로 읽어요.
6. 산출물 — `my_run/liability.csv` · `liability_delta.csv` · `09_liability.png`.

다음 → **[Ch.10 — 랭킹·리포트](../10_ranking_report/10_ranking_lab.ipynb)**